In [ ]:
import os
import torch
import telebot
from transformers import AutoModelForCausalLM, AutoTokenizer

TELEGRAM_TOKEN = "8590599060:AAEYBI4nSmDBu-PDsewUA6IGhaf0gO4-vu8"
LLM_PATH = "FP-KCV/jawani-sealion-gatra-2-9b"

tokenizer = AutoTokenizer.from_pretrained(LLM_PATH)
model = AutoModelForCausalLM.from_pretrained(
    LLM_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

bot = telebot.TeleBot(TELEGRAM_TOKEN)

@bot.message_handler(commands=['start', 'help'])
def send_welcome(message):
    bot.reply_to(message, "Sugeng rahayu. Kula Asisten AI panjenengan. Wonten ingkang saged kula biyantu dinten menika?")

@bot.message_handler(func=lambda message: True)
def handle_message(message):
    user_query = message.text
    chat_id = message.chat.id
    
    bot.send_chat_action(chat_id, 'typing')

    system_instruction = "Panjenengan minangka asisten profesional lan resmi. Tansah nggunakake basa Jawa Krama Lugu ingkang efektif lan lugas kanggo mbiyantu pakaryan."
    prompt = f"Instruksi: {system_instruction}\nPengguna: {user_query}\nAsisten:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    try:
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs, 
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        full_response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        if "Asisten:" in full_response:
            clean_response = full_response.split("Asisten:")[-1].strip()
        else:
            clean_response = full_response.replace(prompt, "").strip()
            
        bot.reply_to(message, clean_response)
        
    except Exception as e:
        print(f"Error: {e}")
        bot.reply_to(message, "Nyuwun sewu, sistem saweg ngalami gangguan teknis.")

bot.infinity_polling()